1

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score
import difflib

train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

for df in [train_df, test_df]:
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)
    
    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)
    
    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates
    
    df['expected_salary'] = df['expected_salary'].fillna(df['expected_salary'].median())
    df['salary_min_usd'] = df['salary_min_usd'].fillna(df['salary_min_usd'].median())
    df['salary_max_usd'] = df['salary_max_usd'].fillna(df['salary_max_usd'].median())
    
    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']
    
    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) / 
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

2

In [ ]:
def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}
def build_advanced_features(df, fitted_idf=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    vals = []
    for a, b in zip(cand_sk, req_sk):
        if not b:
            vals.append(0.0)
            continue
        num = sum(idf.get(t, 0.0) for t in (a & b))
        den = sum(idf.get(t, 0.0) for t in b)
        vals.append(num / den if den else 0.0)
    df["skill_match_idf"] = vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal, np.where(cand_sal > jmax, cand_sal - jmax, 0))

    prev = df["previous_companies"].apply(tokenize_pipe)
    freq = Counter()
    for s in prev:
        freq.update(s)
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)
    
    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    
    if "in_experience_band" in df.columns:
        df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
        
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]
    df["expgrank_x_salgrank"] = df.groupby('job_id')["experience_gap"].rank(pct=True) * df.groupby('job_id')["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * df.groupby('job_id')["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity', 
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband", 
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap"
    ]
    g = df.groupby("job_id")
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
            
    return df, idf

train_df, train_idf = build_advanced_features(train_df)
test_df, _ = build_advanced_features(test_df, fitted_idf=train_idf)

3

In [3]:
def oof_te_highcard(train, test, cat_col, group_col="job_id", label="relevance_label", n_splits=5, k=20, noise=0.01, min_count=1):
    tr = train.copy()
    te = test.copy()
    
    tr[cat_col] = tr[cat_col].fillna("MISSING").astype(str).str.lower().str.strip()
    te[cat_col] = te[cat_col].fillna("MISSING").astype(str).str.lower().str.strip()

    tr["_nl"] = tr.groupby(group_col)[label].transform(lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = tr["_nl"].mean()
    rng = np.random.default_rng(42)
    oof = np.full(len(tr), gmean, dtype=float)
    gkf = GroupKFold(n_splits=n_splits)

    for tr_idx, val_idx in gkf.split(tr, groups=tr[group_col]):
        fold = tr.iloc[tr_idx]
        agg = fold.groupby(cat_col)["_nl"].agg(["mean", "count"])
        agg = agg[agg["count"] >= min_count]
        smooth = (agg["mean"] * agg["count"] + gmean * k) / (agg["count"] + k)
        mapped = tr.iloc[val_idx][cat_col].map(smooth).fillna(gmean).values
        mapped = mapped * (1 + rng.normal(0, noise, size=len(mapped)))
        oof[val_idx] = mapped

    full = tr.groupby(cat_col)["_nl"].agg(["mean", "count"])
    full = full[full["count"] >= min_count]
    full_s = (full["mean"] * full["count"] + gmean * k) / (full["count"] + k)
    test_enc = te[cat_col].map(full_s).fillna(gmean).values
    
    tr_freq = tr[cat_col].map(tr[cat_col].value_counts()).fillna(0)
    te_freq = te[cat_col].map(tr[cat_col].value_counts()).fillna(0)
    
    return oof, test_enc, tr_freq, te_freq

high_card_cols = ['current_title', 'job_location', 'university', 'industry']
for col in high_card_cols:
    tr_enc, te_enc, tr_freq, te_freq = oof_te_highcard(train_df, test_df, col)
    train_df[f"{col}_te"] = tr_enc
    test_df[f"{col}_te"] = te_enc
    train_df[f"{col}_freq"] = tr_freq
    test_df[f"{col}_freq"] = te_freq

3.5

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]
    
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (train_df['expected_salary'].median() + 1)
    
    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)
        
        job_train_tf = tfidf.transform(train_df[col_job].fillna(""))
        cand_train_tf = tfidf.transform(train_df[col_cand].fillna(""))
        job_test_tf = tfidf.transform(test_df[col_job].fillna(""))
        cand_test_tf = tfidf.transform(test_df[col_cand].fillna(""))
        
        svd = TruncatedSVD(n_components=10, random_state=42)
        job_train_svd = svd.fit_transform(job_train_tf)
        cand_train_svd = svd.transform(cand_train_tf)
        job_test_svd = svd.transform(job_test_tf)
        cand_test_svd = svd.transform(cand_test_tf)
        
        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_train_svd, cand_train_svd)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_test_svd, cand_test_svd)
        
        for i in range(10):
            train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
            test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]
            
    return train_df, test_df

train_df, test_df = add_lsa_and_global_features(train_df, test_df)

/tmp/ipykernel_140773/1098272815.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f'{col_cand}_svd_{i}'] = cand_train_svd[:, i]
/tmp/ipykernel_140773/1098272815.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[f'{col_cand}_svd_{i}'] = cand_test_svd[:, i]


4


In [5]:
cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
    'company_size', 'industry', 'current_title', 'skills', 'education_level', 
    'university', 'previous_companies', 'certifications', 'english_proficiency', 
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label'
]
all_features = [c for c in train_df.columns if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

def fast_null_importance_pruner(X, y, groups, features):
    m_actual = lgb.LGBMRanker(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    sort_idx = np.argsort(groups, kind='stable')
    _, counts = np.unique(groups[sort_idx], return_counts=True)
    m_actual.fit(X.iloc[sort_idx], y[sort_idx], group=counts)
    actual_imp = m_actual.booster_.feature_importance(importance_type="gain")
    
    imp_df = pd.DataFrame({'feature': features, 'importance': actual_imp})
    imp_df = imp_df.sort_values('importance', ascending=False)
    keep_features = imp_df.head(50)['feature'].tolist()
    
    return keep_features

final_features = fast_null_importance_pruner(train_df[all_features], train_df['relevance_label'].values, train_df['job_id'].values, all_features)

train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048429 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16708
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 97


5


In [6]:
def ndcg_by_group(y_true, y_pred, groups, k=10):
    df = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k) for _, gr in df.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

X = train_df[final_features]
y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'n_estimators': 1500,
    'learning_rate': 0.03, 'num_leaves': 31, 'max_depth': 6,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.65,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'label_gain': [0, 1, 3, 7, 15],
    'random_state': 42, 'n_jobs': -1
}

job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(5):
    val_jobs = folds[i]
    tr_jobs = np.concatenate([folds[j] for j in range(5) if j != i])
    
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    
    cv_splits.append((tr_idx, val_idx))

oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
    
    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
    
    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)
    
    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])
    
    cm = CatBoostRanker(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)

best_score, best_w = -1, 0.5
for w in np.linspace(0, 1, 21):
    sc = ndcg_by_group(y, w * r_lgb + (1 - w) * r_cat, groups)
    if sc > best_score:
        best_score, best_w = sc, w

_, c_full = np.unique(groups, return_counts=True)

final_lgb = lgb.LGBMRanker(**lgb_params)
final_lgb.fit(X, y, group=c_full)

final_cat = CatBoostRanker(loss_function="YetiRank", iterations=1000, learning_rate=0.03, depth=6, l2_leaf_reg=3, random_seed=42, verbose=0)
final_cat.fit(Pool(X, y, group_id=groups))

X_test = test_df[final_features]
test_groups = test_df['job_id'].values

final_blend = best_w * group_rank(final_lgb.predict(X_test), test_groups) + (1 - best_w) * group_rank(final_cat.predict(X_test), test_groups)

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ensemble_final555.csv', index=False)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020547 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9091
[LightGBM] [Info] Number of data points in the train set: 101108, number of used features: 50
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9082
[LightGBM] [Info] Number of data points in the train set: 95455, number of used features: 50
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.063697 seconds.
You can set `force_row_wise=true` to re